# FPL Features

## Building features with histroical gameweek data from multiple seasons

In [16]:
import pandas as pd

In [17]:
from pathlib import Path

RAW_PATH = Path("../data/raw/merged_gw_2022_2026.parquet")
seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]

if RAW_PATH.exists():
    all_df = pd.read_parquet(RAW_PATH)
    print("Loaded from local file")
else:
    frames = []
    for season in seasons:
        url = f"https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{season}/gws/merged_gw.csv"
        season_df = pd.read_csv(url)
        season_df["season"] = season
        frames.append(season_df)
    all_df = pd.concat(frames, ignore_index=True)
    all_df.to_parquet(RAW_PATH)
    print("Downloaded and saved")

Loaded from local file


In [18]:
all_df["kickoff_time"] = pd.to_datetime(all_df["kickoff_time"])
all_df = all_df.sort_values(["season", "element", "kickoff_time"]).reset_index(drop=True)

In [20]:
group = all_df.groupby(["season", "element"])["minutes"]

all_df["minutes_avg_last_5"] = group.transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

all_df["minutes_avg_last_3"] = group.transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

In [21]:
check = all_df[(all_df["season"] == "2025-26") & (all_df["name"].str.contains("Saka"))]
check[["GW", "kickoff_time", "minutes", "minutes_avg_last_3", "minutes_avg_last_5"]].head(10)

,GW,kickoff_time,minutes,minutes_avg_last_3,minutes_avg_last_5
84405,1,2025-08-17 15:30:00+00:00,90,NaN,NaN
84406,2,2025-08-23 16:30:00+00:00,52,90.000000,90.000000
84407,3,2025-08-31 15:30:00+00:00,0,71.000000,71.000000
84408,4,2025-09-13 11:30:00+00:00,0,47.333333,47.333333
84409,5,2025-09-21 15:30:00+00:00,45,17.333333,35.500000
84410,6,2025-09-28 15:30:00+00:00,69,15.000000,37.400000
84411,7,2025-10-04 14:00:00+00:00,90,38.000000,33.200000
84412,8,2025-10-18 16:30:00+00:00,90,68.000000,40.800000
84413,9,2025-10-26 14:00:00+00:00,65,83.000000,58.800000
84414,10,2025-11-01 15:00:00+00:00,90,81.666667,71.800000


In [22]:
ROLLING_COLS = ["minutes", "total_points", "expected_goals", "expected_assists", "bonus"]
all_df[ROLLING_COLS].dtypes

minutes               int64
total_points          int64
expected_goals      float64
expected_assists    float64
bonus                 int64
dtype: object

In [24]:
def add_rolling_features(df, cols, windows):
    """For each column and window size, add the player's average over their previous matches."""
    df = df.copy()
    grouped = df.groupby(["season", "element"])

    for col in cols:
        for w in windows:
            new_name = f"{col}_avg_last_{w}"
            df[new_name] = grouped[col].transform(
                lambda s: s.shift(1).rolling(w, min_periods=1).mean()
            )
    return df

In [25]:
WINDOWS = [3, 5]
features_df = add_rolling_features(all_df, ROLLING_COLS, WINDOWS)

new_cols = [c for c in features_df.columns if "_avg_last_" in c]
print(len(new_cols))
new_cols

10


['minutes_avg_last_5',
 'minutes_avg_last_3',
 'total_points_avg_last_3',
 'total_points_avg_last_5',
 'expected_goals_avg_last_3',
 'expected_goals_avg_last_5',
 'expected_assists_avg_last_3',
 'expected_assists_avg_last_5',
 'bonus_avg_last_3',
 'bonus_avg_last_5']

In [ ]:
saka = features_df[(features_df["season"] == "2025-26") & (features_df["name"].str.contains("Saka"))]
saka[["GW", "total_points", "total_points_avg_last_3"]].head(6)